<a href="https://colab.research.google.com/github/cqx931/AsWeMaySpeak/blob/main/week6/RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **RAG Chatbot**

![](https://developers.llamaindex.ai/python/_astro/basic_rag.sdlwNwWz_Z1yQWLG.webp)



In this notebook we’re going to walk through the process of building a RAG pipeline using Ollama and LlamaIndex.
The idea is that to make use of open-source LLMs(host them locally) and use them to do Q&As with our own dataset.

---
## **Running llms using Ollama**

First, use the following code cell to install ollama and cuda-drivers on Google Colab. This is going to take a while.(5-6min)

In [ ]:
# @title Install components
!curl https://ollama.ai/install.sh | sh
!pip install ollama

!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get update && sudo apt-get install -y cuda-drivers

import os
# Set LD_LIBRARY_PATH so the system NVIDIA library
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  41504      0 --:--:-- --:--:-- --:--:-- 41503
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [345 B]
Get:5 https://develope

Now we can start the ollama server.

In [ ]:
# @title Start server
import subprocess
proccess = subprocess.Popen(['ollama', 'serve'])

In [ ]:
# @title Select your model
model = "alibayram/smollm3"
!ollama pull {model}

In [ ]:
# @title Interacting with the model
question = "hello" # @param {"type":"string"}
from IPython.display import display, Markdown
import ollama
response = ollama.chat(model, messages=[
  {
    'role': 'user',
    'content': question,
  },
])
#print(response['message']['content'])
display(Markdown(response['message']['content']))

<think>

</think>
Hello! How can I help you today?

---
## LlamaIndex

LlamaIndex is a framework for building LLM applications, connecting diverse datasets to LLMs.

For our usage, you will need to install `llama-index`, `llama-index-llms-ollama` and `llama-index-llms-huggingface` using pip.


In [ ]:
!pip install llama-index llama-index-llms-ollama llama-index-embeddings-huggingface

INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

Now let's create an llm instance inside llama_index using ollama. For this notebook, we will be using the smolllm3 model. You can also switch it to other models that you have downloaded locally to your machine or models from OpenAI with your own API. We are also setting the request_timeout parameter here, it sets the maximum waiting time for a response.

In [ ]:
from llama_index.llms.ollama import Ollama

llm = Ollama(model="alibayram/smollm3", request_timeout=60.0)


We can set our system prompt and ask a question using the following code.

In [ ]:
# @title Chat with Ollama models using Llama_index
from llama_index.core.llms import ChatMessage
from IPython.display import display, Markdown
messages = [
    ChatMessage(role="system", content="You are a helpful assistant."),
    ChatMessage(role="user", content="Where is the capital of Germany?"),
]
response = llm.chat(messages)
display(Markdown(f'<p style="font-size:18px">{response}</p>'))

<p style="font-size:18px">assistant: <think>

</think>
The capital of Germany is Berlin. It's located in the northeastern part of the country, near the Baltic Sea and Poland. Berlin has been the capital since 1990 when it was reunited with West Berlin after the Cold War. Before that, the city served as the capital for East Germany from 1949 to 1961.</p>

## Load Data and setup the Embedding model

To Train a RAG Pipeline, we will need to train some indexes using embedding models. Here we are using a very small and basic one that is suggested by LlamaIndex. If you want to improve the quality of your chatbot, you can consider using larger embedding models or embedding models that work better in the language of your text.


In [ ]:
# @title Load Embedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# @title Change Settings
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model

## Load Data

For the demo, we are going to use exhibition data from MoMA from 1980 to 1989. The data is a subset of [the original data](https://github.com/MuseumofModernArt/exhibitions).

In [ ]:
! curl -O https://raw.githubusercontent.com/cqx931/AsWeMaySpeak/refs/heads/main/week6/MoMAExhibitions_filtered.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1163k  100 1163k    0     0  2764k      0 --:--:-- --:--:-- --:--:-- 2763k


I cleaned the original data with the code below to make the file size smaller and faster to train. I have included the code here in case this can be useful for you to filter your own dataset.

In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("MoMAExhibitions1929to1989.csv")

# Clean column names (remove extra spaces)
df.columns = df.columns.str.strip()

# Convert the date column to datetime
df["ExhibitionBeginDate"] = pd.to_datetime(df["ExhibitionBeginDate"], errors="coerce")

# Filter dates after Jan 1, 1980
df_filtered = df[df["ExhibitionBeginDate"] > "1980-01-01"]

# Columns to drop
cols_to_drop = [
    "ExhibitionNumber",
    "ExhibitionSortOrder",
    "ExhibitionCitationDate",
    "Suffix",
    "Institution",
    "ConstituentBeginDate",
    "ConstituentEndDate",
    "ConstituentID",
    "VIAFID",
    "WikidataID",
    "ULANID"
]

# Drop columns (only if they exist)
df_filtered = df_filtered.drop(columns=[c for c in df_filtered.columns if c in cols_to_drop])

# Save to a new CSV
df_filtered.to_csv("filtered_exhibitions_new.csv", index=False)

print("Filtered CSV saved as filtered_exhibitions.csv") #1.8MB

We are going to use `SimpleDirectoryReader` to load the files. As the name suggested, you can also read a whole directory of data in multiple files with it.

In [ ]:
from llama_index.core import SimpleDirectoryReader

data = SimpleDirectoryReader(input_files=["MoMAExhibitions_filtered.csv"]).load_data()
#

---
## Indexing

An Index is a data structure that allows us to quickly retrieve relevant context for a user query. For LlamaIndex, it’s the core foundation for RAG pipelines. The most commonly used is the `VectorStoreIndex`, which is also the one we are going to use here.

When LlamaIndex takes in a document, it internally parses/chunks them into Node objects. By default the chunk size is 1024 tokens, with a default overlap of 20 tokens.

There are also other types of indexes supported by LlamaIndex, you can check them out [here](https://developers.llamaindex.ai/python/framework/module_guides/indexing/index_guide/)

In [ ]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(data, show_progress=True)
# convert the index to a query engine
vector_query_engine = vector_index.as_query_engine(similarity_top_k=5)
# default similarity_top_k is 2

Parsing nodes:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/572 [00:00<?, ?it/s]

![alt vector_store_index](https://developers.llamaindex.ai/python/_astro/vector_store_query.DiFHQid7_EaGmz.webp)
In a `VectorStoreIndex`, a piece of text is usually chunked into different nodes and being indexed in the vector store. When a user sends a query, this query will be represented by a sentence embedding, and it is used to find nodes that are related to the question. If `similarity_top_k=2`, then two nodes are returned, and the query engine will try to synthesize a response based on the information from the given nodes.

Notice that we have changed one parameter here to improve the quality of our query result.
- similarity_top_k: The larger the value, the more context a query gets. But it also makes your model responde slower and could potentially introduce unnecessary noise.

You can try to adjust similarity_top_k to different values to see its influence on the generative result of the same question.

In [ ]:
response = vector_query_engine.query("What exhibitions were held in 1988?")

display(Markdown(f'<p style="font-size:12px">{response}</p>'))


<p style="font-size:12px"><think>

</think>
In 1988, several notable exhibitions at MoMA included "The Modern Poster" which was held from September 6 to June 6. This exhibition featured artists such as Tom Purvis, Louis Raemaekers, Gunter Rambow, Paul Rand, Johannes Regn, Eduard Renggli, Aleksandr Rodchenko, Takao Sasai, and Koichi Sato.</p>


---
## **Exercise 1**
Try to ask the model different questions about the dataset.
- What is it good at and what doesn't work so well? What could be potential ways to improve the quality?
- What happens when you ask something out of the scope of the data?

In [ ]:
response = vector_query_engine.query("Which artists are included in the exhibition 'Around Picasso'?")

display(Markdown(f'<p style="font-size:12px">{response}</p>'))


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

*If you get error message like "Failed to connect to Ollama.", you need to go to the cell at the beginning to restart the Ollama server.

----
## **Chat Engine**

So far what we have built is just a simple query engine. It's not a full chatbot yet. We can make it a chat bot though by passing a memory buffer to it and init it as a `chat_engine`

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

memory = ChatMemoryBuffer.from_defaults(token_limit=4096)


In [ ]:
chat_engine = vector_index.as_chat_engine(
    chat_mode="context", # other modes: "condense_question", "condense_plus_context"
    memory=memory,
    similarity_top_k=2,
    system_prompt=(
         "You are an assistant who provides useful information for exhibitions at MoMA"
     )
)

print("RAG Chat with Memory. Type 'exit' to quit.\n")

while True:
    q = input("You: ")
    if q.strip().lower() == "exit":
        break

    response = chat_engine.chat(q)
    print("Assistant:", response, "\n")

RAG Chat with Memory. Type 'exit' to quit.

You: hi
Assistant: <think>

</think>
Hello again! How can I assist you today? Do you have any questions about MoMA exhibitions data, need help with something else related to it, or perhaps want to explore modern art further? Let me know how I can be of assistance. 

You: what did we talked about before
Assistant: <think>

</think>
Before our conversation, you asked for recommendations on exhibitions at the Museum of Modern Art (MoMA) that focus on modern art. I provided a list of five exhibitions that showcase various aspects of modern art and its history.

If you'd like to revisit or explore any specific points from our discussion, feel free to ask! 

You: exit


---
## **Exercise 2**
- Try to replace the dataset with other data you have in mind and see how it works.
- You can also adjust the system prompt to make it a less "helpful" assistant

---


## **From Chat Agent to Agentic Chat**
The general trend at the moment is Multi-Agent AI Systems, but we won't cover it within the scope of the class.

You can build agents with different specialisations with Llama Index, currently it supports `FunctionAgents`, `ReActAgents` and `CodeActAgents`. Each Agent can have access to multiple tools that it uses to generate a response.

If you are interested, you can read more about it [here](https://www.llamaindex.ai/blog/introducing-llama-agents-a-powerful-framework-for-building-production-multi-agent-ai-systems).

---

## **Reference**
- [Ollama Runner](https://colab.research.google.com/github/tecepeipe/ollama-colab-runner/blob/main/ollama_colab_runner.ipynb#scrollTo=O5toc_VkVffm)
- [Official Documentation of LlamaIndex](https://developers.llamaindex.ai/python/framework/)